# Bronze Layer ETL Pipeline
Digital Lifestyle Benchmark Dataset — Medallion Architecture

This notebook implements the Bronze Layer for the Digital Lifestyle Benchmark dataset.

In this stage, the raw data is loaded as-is without cleaning or transformations. Column names are standardized, exact duplicate rows are removed

The final output is saved as:

Bronze_Layer/Digital_Lifestyle_Benchmark_Bronze.csv

This file serves as the source for the Silver Layer, where data cleaning and further processing will be performed.

## 1. Import Required Libraries

In [ ]:
# pandas -> data loading, manipulation and CSV I/O
import pandas as pd

# os -> folder creation and file path handling (Bronze_Layer directory)
import os

# re -> regular expressions, used for robust column name standardization
import re



## 2. Define File Paths & Create the Bronze Layer Folder

All configurable values (input file, output folder, output file name) live in
one place so the notebook is easy to re-point at a different raw file.

In [ ]:
# Path to the raw CSV file exactly as received from the source system.
# Update this if the raw file lives in a different folder.
RAW_FILE_PATH = "/content/digital_lifestyle_benchmark_2025_100k.csv"

# The raw file uses ';' as its field delimiter (confirmed by inspecting the
# header row). Keep this as a variable so it is explicit and easy to change.
RAW_FILE_DELIMITER = ";"

# Name of the Bronze Layer output folder (created in the current working
# directory if it does not already exist).
BRONZE_FOLDER = "Bronze_Layer"

# Name of the Bronze Layer output file.
BRONZE_FILE_NAME = "Digital_Lifestyle_Benchmark_Bronze.csv"

# Full output path = folder + file name.
BRONZE_FILE_PATH = os.path.join(BRONZE_FOLDER, BRONZE_FILE_NAME)

# Create the Bronze_Layer folder if it doesn't exist yet.
# exist_ok=True prevents an error if the folder is already there.
os.makedirs(BRONZE_FOLDER, exist_ok=True)

print(f"Bronze Layer folder ready at: ./{BRONZE_FOLDER}/")

Bronze Layer folder ready at: ./Bronze_Layer/


## 3. Load the Raw Dataset

The dataset is loaded exactly as it exists in the source file. No rows or
values are altered at this stage.

In [ ]:
# Load the raw CSV into a pandas DataFrame using the source delimiter.
# This is a straight load — no filtering, casting, or value changes.
df_raw = pd.read_csv(RAW_FILE_PATH, sep=RAW_FILE_DELIMITER)

print(f"Raw file loaded: {RAW_FILE_PATH}")
print(f"Raw shape (rows, columns): {df_raw.shape}")

Raw file loaded: /content/digital_lifestyle_benchmark_2025_100k.csv
Raw shape (rows, columns): (100000, 24)


## 4. Preserve the Original Data (No Cleaning or Modifications)

A working copy (`df`) is created from `df_raw` so the raw DataFrame stays
untouched as a reference. In the Bronze Layer we deliberately do **not**:
- Fill, drop, or impute missing values.
- Change data types.
- Filter, transform, or recalculate any column values.

The only operations permitted at this stage are structural (column naming),
deduplication of exact duplicate rows, and the addition of metadata columns.

In [ ]:
# Work on a copy so df_raw remains an untouched snapshot of the source data.
df = df_raw.copy()

# Quick confirmation that the values themselves have not changed yet.
print("Working copy created. Column values are identical to the raw source.")
print(df.dtypes)

Working copy created. Column values are identical to the raw source.
id                            int64
age                           int64
gender                       object
region                       object
income_level                 object
education_level              object
daily_role                   object
device_hours_per_day        float64
phone_unlocks                 int64
notifications_per_day         int64
social_media_mins             int64
study_mins                    int64
physical_activity_days        int64
sleep_hours                 float64
sleep_quality               float64
anxiety_score               float64
depression_score              int64
stress_level                float64
happiness_score             float64
focus_score                 float64
high_risk_flag                int64
device_type                  object
productivity_score          float64
digital_dependence_score    float64
dtype: object


## 5. Standardize Column Names

Column names are standardized for compatibility with Power BI and Python
(no leading/trailing spaces, no embedded spaces). This does **not** touch any
cell values — only the header row.

In [ ]:
def standardize_column_name(col_name: str) -> str:
    """
    Standardize a single column name:
    1. Strip leading/trailing whitespace.
    2. Replace any run of internal whitespace with a single underscore.
    This keeps names valid as Python identifiers / DataFrame attributes and
    avoids the spaces-in-column-names issues Power BI sometimes runs into.
    """
    col_name = str(col_name).strip()           # remove leading/trailing spaces
    col_name = re.sub(r"\s+", "_", col_name)    # spaces -> underscores
    return col_name

# Keep the original column order, just clean each label.
original_columns = df.columns.tolist()
df.columns = [standardize_column_name(c) for c in df.columns]

print("Column names standardized.")
print(df.columns.tolist())

Column names standardized.
['id', 'age', 'gender', 'region', 'income_level', 'education_level', 'daily_role', 'device_hours_per_day', 'phone_unlocks', 'notifications_per_day', 'social_media_mins', 'study_mins', 'physical_activity_days', 'sleep_hours', 'sleep_quality', 'anxiety_score', 'depression_score', 'stress_level', 'happiness_score', 'focus_score', 'high_risk_flag', 'device_type', 'productivity_score', 'digital_dependence_score']


## 6. Remove Exact Duplicate Rows Only

Only rows that are **100% identical across every column** are removed. This is
a structural deduplication step, not a data quality / cleaning step — partial
matches or "near duplicates" are intentionally left untouched.

In [ ]:
rows_before = len(df)

# keep="first" retains the first occurrence of each duplicated row and drops
# the rest. A row only counts as a duplicate if every column value matches.
df = df.drop_duplicates(keep="first").reset_index(drop=True)

rows_after = len(df)
duplicate_rows_removed = rows_before - rows_after

print(f"Rows before deduplication: {rows_before}")
print(f"Rows after deduplication:  {rows_after}")
print(f"Exact duplicate rows removed: {duplicate_rows_removed}")

Rows before deduplication: 100000
Rows after deduplication:  100000
Exact duplicate rows removed: 0


## 7. Data Quality Summary (Bronze Audit Log)

In [ ]:
# Total missing values across the Bronze dataset (reported, NOT imputed).
total_missing_values = df.isnull().sum().sum()

print("===== BRONZE LAYER SUMMARY =====")
print(f"Rows:                     {df.shape[0]}")
print(f"Columns:                  {df.shape[1]}")
print(f"Duplicate rows removed:   {duplicate_rows_removed}")
print(f"Total missing values:     {total_missing_values}")
print("=================================")

===== BRONZE LAYER SUMMARY =====
Rows:                     100000
Columns:                  24
Duplicate rows removed:   0
Total missing values:     0


## 8. Save the Bronze Layer Dataset

The final Bronze DataFrame (standardized headers + metadata columns, raw
values untouched) is written to `Bronze_Layer/Digital_Lifestyle_Benchmark_Bronze.csv`.

In [ ]:
# index=False avoids writing the pandas row index as an extra column.
df.to_csv(BRONZE_FILE_PATH, index=False)

print(f"Bronze Layer file saved to: {BRONZE_FILE_PATH}")

Bronze Layer file saved to: Bronze_Layer/Digital_Lifestyle_Benchmark_Bronze.csv


## Summary & Next Steps

The Bronze Layer is complete:
- Raw data ingested with zero changes to actual values.
- Column headers standardized for Power BI / Python compatibility.
- Exact duplicate rows removed.
- Output persisted to `Bronze_Layer/Digital_Lifestyle_Benchmark_Bronze.csv`.

**Next stage — Silver Layer** would typically: enforce data types, handle
missing values and outliers, apply business rules, and conform categorical
values — all without touching this Bronze file, which remains the immutable
source of truth for re-processing.